# Module 4 — Season Projections and Draft Rankers

This notebook aggregates per-game predictions into player-season projections and produces two ranked lists: a season total (sum) for full-season drafts, and a per-game average (mean, filtered to a thirty-game minimum) for mid-season decisions. Collapsing both into a single ranking would conflate two different questions — durability versus per-game value.

The core evaluation metric here is Spearman rank correlation, not MAE, because the business question is ordering, not point estimates. For a draft board, whether Jokić is projected at 60 FP or 65 FP does not matter; whether he is projected above or below the next player does.

**Finding:** the baseline produced a slightly higher rank correlation (0.994) than LightGBM (0.990). At season-level aggregation, the law of large numbers washes out the model's per-game precision advantage. This is one of the most interview-relevant findings in the project — rank correlation and MAE answer different questions.

## Setup — reproduce Module 3's model

In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# Load feature-engineered data
df = pd.read_parquet('data/processed/model_df.parquet')

feature_cols = ['rest_days_capped', 'returned_from_absence',
                'rolling_fp_5', 'rolling_fp_10',
                'is_home', 'opp_fp_allowed', 'rolling_min_10',
                'rolling_fp_std_10', 'fp_momentum']

clean_df = df.dropna(subset=feature_cols).copy()

# Same chronological split as Module 3
train = clean_df[clean_df['GAME_DATE'] < '2024-10-01'].copy()
test  = clean_df[clean_df['GAME_DATE'] >= '2024-10-01'].copy()

# Train LightGBM (same hyperparameters as Module 3)
model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    random_state=42, verbose=-1,
)
model.fit(train[feature_cols], train['FP'])

# Predict 2024-25 games
test['predicted_FP'] = model.predict(test[feature_cols])

print(f'Train: {train.shape[0]:,} rows  |  Test: {test.shape[0]:,} rows')
print(f'Predictions range: {test["predicted_FP"].min():.1f} to {test["predicted_FP"].max():.1f}')
test[['PLAYER_NAME', 'GAME_DATE', 'FP', 'predicted_FP']].head()

Train: 70,843 rows  |  Test: 25,332 rows
Predictions range: 1.2 to 69.9


,PLAYER_NAME,GAME_DATE,FP,predicted_FP
183,LeBron James,2024-10-22,32,49.534990
184,LeBron James,2024-10-25,37,49.404945
185,LeBron James,2024-10-26,66,42.483490
186,LeBron James,2024-10-28,23,51.790738
187,LeBron James,2024-10-30,32,53.780959


In [9]:
projections = test.groupby(['PLAYER_ID', 'PLAYER_NAME']).agg(
    games_played=('GAME_DATE', 'count'),
    actual_total_fp=('FP', 'sum'),
    predicted_total_fp=('predicted_FP', 'sum'),
    actual_avg_fp=('FP', 'mean'),
    predicted_avg_fp=('predicted_FP', 'mean'),
).reset_index()

print(f'Players: {projections.shape[0]}')
projections.head()

Players: 530


,PLAYER_ID,PLAYER_NAME,games_played,actual_total_fp,predicted_total_fp,actual_avg_fp,predicted_avg_fp
0,2544,LeBron James,70,3431,3401.075183,49.014286,48.586788
1,101108,Chris Paul,82,2519,2446.287582,30.719512,29.832775
2,200768,Kyle Lowry,35,519,590.685466,14.828571,16.876728
3,200782,P.J. Tucker,3,25,26.210545,8.333333,8.736848
4,201142,Kevin Durant,62,2837,2655.281276,45.758065,42.827117


In [11]:
# Rank players by projected total FP
draft_board = projections.sort_values('predicted_total_fp', ascending=False).reset_index(drop=True)
draft_board['rank'] = draft_board.index + 1

# Top 20 draft picks according to our model
draft_board[['rank', 'PLAYER_NAME', 'games_played', 
             'predicted_total_fp', 'actual_total_fp']].head(20)

,rank,PLAYER_NAME,games_played,predicted_total_fp,actual_total_fp
0,1,Shai Gilgeous-Alexander,76,4109.475118,4460
1,2,Nikola Jokić,70,4077.730404,4879
2,3,Giannis Antetokounmpo,67,3636.076556,3842
3,4,LeBron James,70,3401.075183,3431
4,5,Jayson Tatum,72,3355.892552,3482
5,6,Anthony Edwards,79,3325.641910,3484
6,7,Tyrese Haliburton,73,3317.283133,3470
7,8,James Harden,79,3297.069694,3571
8,9,Trae Young,76,3284.553683,3443
9,10,Cade Cunningham,70,3228.494709,3412


In [12]:
draft_board = (projections[projections['games_played'] >= 30]
               .sort_values('predicted_avg_fp', ascending=False)
               .reset_index(drop=True))
draft_board['rank'] = draft_board.index + 1

# Top 20 draft picks according to our model
draft_board[['rank', 'PLAYER_NAME', 'games_played', 
             'predicted_avg_fp', 'actual_avg_fp']].head(20)

,rank,PLAYER_NAME,games_played,predicted_avg_fp,actual_avg_fp
0,1,Nikola Jokić,70,58.253291,69.700000
1,2,Victor Wembanyama,46,56.652217,57.434783
2,3,Giannis Antetokounmpo,67,54.269799,57.343284
3,4,Shai Gilgeous-Alexander,76,54.072041,58.684211
4,5,Anthony Davis,51,51.132420,51.862745
5,6,Luka Dončić,50,50.257924,53.060000
6,7,LeBron James,70,48.586788,49.014286
7,8,Jayson Tatum,72,46.609619,48.361111
8,9,Cade Cunningham,70,46.121353,48.742857
9,10,Tyrese Haliburton,73,45.442235,47.534247


In [13]:
# Spearman rank correlation between predicted and actual per-game rank
rank_correlation = draft_board[['predicted_avg_fp', 'actual_avg_fp']].corr(method='spearman').iloc[0, 1]
print(f'Rank correlation (per-game, 30+ games): {rank_correlation:.3f}')

Rank correlation (per-game, 30+ games): 0.990


In [14]:
# Also aggregate baseline predictions per player
test['baseline_pred'] = test['rolling_fp_10']
baseline_projections = test.groupby(['PLAYER_ID', 'PLAYER_NAME']).agg(
    games_played=('GAME_DATE', 'count'),
    actual_avg_fp=('FP', 'mean'),
    baseline_avg_fp=('baseline_pred', 'mean'),
).reset_index()

baseline_qualified = baseline_projections[baseline_projections['games_played'] >= 30]
baseline_rank_corr = baseline_qualified[['baseline_avg_fp', 'actual_avg_fp']].corr(method='spearman').iloc[0, 1]

print(f'Baseline rank correlation (30+ games): {baseline_rank_corr:.3f}')
print(f'LightGBM rank correlation:             0.990')

Baseline rank correlation (30+ games): 0.994
LightGBM rank correlation:             0.990
